In [1]:
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.append(str(root))

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

from src.data_preprocessing import(
    load_and_clean_data,
    stratified_split,
)

from src.pipeline import build_pipeline

In [3]:
Data_path = root / "data" /"raw" / "cardata.csv"

df = load_and_clean_data(Data_path)
print(f"dataset shape: {df.shape}")
df.head()

dataset shape: (2058, 20)


,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0


In [4]:
train_set, test_set = stratified_split(df)
print("Trainng set: ", train_set.shape)
print("Test set:", test_set.shape)

Trainng set:  (1646, 20)
Test set: (412, 20)


In [5]:
x_train = train_set.drop(columns="Price")
y_train = train_set["Price"]

x_test = test_set.drop(columns="Price")
y_test = test_set["Price"]

In [6]:
baseline_model = Pipeline([
    ("preprocessing",build_pipeline(),),
    ("model",RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),),

])

baseline_model.fit(x_train,y_train)
baseline_predictions = baseline_model.predict(
    x_test
)

In [7]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions,
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions,
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions,
)

print("Baseline Model")
print("=" *40)

print(f"mae: {baseline_mae:,.0f}")
print(f"rmse: {baseline_rmse:,.0f}")
print(f"r2: {baseline_r2:,.0f}")

Baseline Model
mae: 259,069
rmse: 722,302
r2: 1


In [8]:
model_pipeline = Pipeline([
("preprocessing",build_pipeline(),),
("model",RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
),),
])

param_distributions={
    "model__n_estimators":[
        200,300,400,500,600
    ],
    "model__max_depth":[
        None,10,20,30,40,
    ],
    "model__min_samples_split":[
        2,5,10,
    ],
    "model__min_samples_leaf":[
        1,2,4,
    ],
    "model__max_features" :[
        0.5,0.75,1.0,
    ],
}

search = RandomizedSearchCV(
    estimator=model_pipeline,

    param_distributions=param_distributions,

    n_iter=25,

    scoring="neg_root_mean_squared_error",

    cv=5,

    random_state=42,

    n_jobs=-1,

    verbose=2,
)

search.fit(x_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': [0.5, 0.75, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best e

In [9]:
print("Best parameters")
print("=" *40)

for parameter, value in search.best_params_.items():
    print(f"{parameter}:{value}")


print()
print(f"best cv rmse:" f"{-search.best_score_:,.0f}")

Best parameters
model__n_estimators:200
model__min_samples_split:2
model__min_samples_leaf:1
model__max_features:1.0
model__max_depth:None

best cv rmse:1,008,715


In [10]:
tuned_model = search.best_estimator_
tuned_predictions = tuned_model.predict(
    x_test
)

In [11]:
tuned_mae=mean_absolute_error(
    y_test,
    tuned_predictions,
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tuned_predictions,
    )
)

tuned_r2=r2_score(
    y_test,
    tuned_predictions,
)

print("tuned model")
print("="*40)

print(f"mae: {tuned_mae:,.0f}")
print(f"rmse: {tuned_rmse:,.0f}")
print(f"r2_score: {tuned_r2:,.4f}")

tuned model
mae: 247,174
rmse: 656,360
r2_score: 0.9058


In [12]:
comparison = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²",
    ],

    "Current Model": [
        baseline_mae,
        baseline_rmse,
        baseline_r2,
    ],

    "Tuned Model": [
        tuned_mae,
        tuned_rmse,
        tuned_r2,
    ],
})

comparison

,Metric,Current Model,Tuned Model
0,MAE,259068.674254,247174.211177
1,RMSE,722301.997665,656360.074903
2,R²,0.885901,0.905783
